# BERT vs RoBERTa-base — fine-tuning na LIAR

Pobieramy przygotowany dataset z Drive (parquety zapisane przez notatnik preprocessingowy):

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/fakenews_cache'
TOK_DIR   = os.path.join(CACHE_DIR, 'tokenized')
HF_CACHE  = os.path.join(CACHE_DIR, 'hf_cache')
os.environ['HF_HOME'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE

print('TOK_DIR:', os.listdir(TOK_DIR))

In [ ]:
%pip install -q transformers torch scikit-learn

Ladujemy parquety z surowymi splitami (statement + label) - BERT i RoBERTa uzywaja roznych tokenizatorow, wiec tokenizujemy per-model.

In [ ]:
import pandas as pd

VERSION = '2cl'

train = pd.read_parquet(os.path.join(TOK_DIR, f'df_train_{VERSION}.parquet'))
valid = pd.read_parquet(os.path.join(TOK_DIR, f'df_valid_{VERSION}.parquet'))
test  = pd.read_parquet(os.path.join(TOK_DIR, f'df_test_{VERSION}.parquet'))

print('train:', train.shape, ' valid:', valid.shape, ' test:', test.shape)
print('rozklad train:', train['label'].value_counts().to_dict())

## Fine-tuning BERT vs RoBERTa-base

Trenujemy obie sieci z tym samym pipeline. Wybor najlepszej epoki po walidacyjnym macro-F1.

In [ ]:
import copy
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


class LiarDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            'input_ids':      self.enc['input_ids'][i],
            'attention_mask': self.enc['attention_mask'][i],
            'labels':         self.labels[i]
        }


def make_loader(df, tokenizer, shuffle, batch_size=16, max_length=128):
    enc = tokenizer(list(df['statement']), padding='max_length',
                    truncation=True, max_length=max_length, return_tensors='pt')
    ds = LiarDataset(enc, torch.tensor(df['label'].tolist()))
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def predict(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            preds += logits.argmax(1).cpu().tolist()
    return np.array(preds)

In [ ]:
def finetune(model_name, epochs=3, lr=2e-5):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    num_labels = 2 if VERSION == '2cl' else 6
    model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                                num_labels=num_labels).to(device)

    train_loader = make_loader(train, tokenizer, shuffle=True)
    valid_loader = make_loader(valid, tokenizer, shuffle=False)
    test_loader  = make_loader(test,  tokenizer, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_f1, best_state = -1.0, None
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            loss.backward()
            optimizer.step()

        valid_f1 = f1_score(valid['label'].values, predict(model, valid_loader), average='macro')
        print(f'{model_name} | epoch {epoch + 1}/{epochs} | valid_f1={valid_f1:.4f}')
        if valid_f1 > best_f1:
            best_f1, best_state = valid_f1, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    preds = predict(model, test_loader)
    trues = test['label'].values
    acc = accuracy_score(trues, preds)
    p, r, f1, _ = precision_recall_fscore_support(trues, preds, average='macro', zero_division=0)
    print(f'{model_name}: acc={acc:.4f} precision={p:.4f} recall={r:.4f} f1={f1:.4f} '
          f'(best valid_f1={best_f1:.4f})')
    print(classification_report(trues, preds, zero_division=0))
    print('Confusion matrix:')
    print(confusion_matrix(trues, preds))
    return {'name': model_name, 'accuracy': acc, 'precision': p, 'recall': r,
            'f1': f1, 'preds': preds}

In [ ]:
res_bert = finetune('bert-base-uncased')

In [ ]:
res_roberta = finetune('roberta-base')

In [ ]:
results = [res_bert, res_roberta]

ranking = (pd.DataFrame(results)[['name', 'accuracy', 'precision', 'recall', 'f1']]
           .sort_values('f1', ascending=False)
           .round(4)
           .reset_index(drop=True))
ranking